In [1]:
import sys
sys.path.append('../distance-py/build')
import distancepy

import matplotlib.pyplot as plt 
import mujoco
import numpy as np
import quaternion
from tqdm import tqdm

from belief_dynamics_learning.robot import Robot
from belief_dynamics_learning.trajectory_sampling import TrajectorySampler

%load_ext autoreload
%autoreload 2

In [2]:
model = mujoco.MjModel.from_xml_path("../mujoco_franka_emika_panda/scene_box.xml")
box_dims = np.array([0.067, 0.14, 0.095])
pos_obj_init = np.array([0.5, 0.0, 0.0475]) 
orient_obj_init = quaternion.as_float_array(quaternion.from_rotation_vector(np.array([0, 0, 0])))
pose_obj_init = np.concatenate((pos_obj_init, orient_obj_init))
q_home = np.array([0, 0.7, 0, -1.57079, 0, 1.57079+0.7, 0.7853, 0.04])
q_start = np.array([-0.0113, -0.2576, -0.0366, -2.6312, -0.0391, 2.3982, 0.7856])
q_start_9 = np.concatenate([q_start, [0, 0]])
dt_control = 0.01 
dt_mujoco = 0.01 
model.opt.timestep = dt_mujoco
num_particles = 10
system = Robot(model, q_start_9[:8], pose_obj_init, num_particles=num_particles)
system.EE_name = 'grasp_frame'
system.launch_viewer(right_ui=False)

In [3]:
xd = pos_obj_init.copy()
TrajSampler = TrajectorySampler()

In [4]:
q_traj_list = TrajSampler.generate_trajs(q_start, system, xd.reshape(1, -1))
q_traj = q_traj_list[0]
print(q_traj.shape)

(1415, 7)


In [5]:
from belief_dynamics_learning.mujoco_utils import sim_and_show_candidate_traj
sim_and_show_candidate_traj(system, q_traj)

In [10]:
from belief_dynamics_learning.mujoco_utils import compute_gt_rollout
gt_rollout = compute_gt_rollout(system, q_traj, pose_obj_init) 

In [11]:
print(gt_rollout.shape)

(1, 1415, 31)
